In [2]:
import pandas as pd
df = pd.read_csv('/content/clean_data.csv')
print("Data imported successfully. First 5 rows:")
print(df.head())

Data imported successfully. First 5 rows:
                author          published_at  like_count  \
0            @Ipank049  2026-03-15T19:00:49Z           0   
1           @Tmsaja873  2026-03-15T11:15:42Z           0   
2  @fakhrieofisial4819  2026-03-15T07:36:34Z           0   
3    @sudaryadidar1358  2026-03-14T08:23:35Z           0   
4         @KokGiniAmat  2026-03-14T06:33:38Z           0   

                                             comment   public  \
0  Penguasa dan Pelaksana Program MBG menjadi Pen...  Negatif   
1            anaknya makan 1x bapaknya penganguran..  Negatif   
2  Rusak lah negara kalau Nepotisme, korupsi sema...  Negatif   
3  Programpresiden jokowi. Berobat gratid banyak ...  Negatif   
4                               Maling Berkedok Gizi  Negatif   

                                          clean_text  \
0  penguasa dan pelaksana program mbg menjadi pen...   
1               anaknya makan x bapaknya penganguran   
2  rusak lah negara kalau nepotisme ko

In [3]:
X = df['clean_text']
y = df['label']

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

# Fill NaN values in X with an empty string before transformation
X_cleaned = X.fillna('')
X_tfidf = tfidf.fit_transform(X_cleaned)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
print("Naive Bayes:", acc_nb)

Naive Bayes: 0.9462616822429907


In [8]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

acc_svm = accuracy_score(y_test, y_pred_svm)
print("SVM:", acc_svm)

SVM: 0.9766355140186916


In [9]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression:", acc_lr)

Logistic Regression: 0.9587227414330218


In [10]:
from sklearn.metrics import classification_report

print("\n=== NAIVE BAYES ===")
print(classification_report(y_test, y_pred_nb))

print("\n=== SVM ===")
print(classification_report(y_test, y_pred_svm))

print("\n=== LOGISTIC ===")
print(classification_report(y_test, y_pred_lr))


=== NAIVE BAYES ===
              precision    recall  f1-score   support

     negatif       0.00      0.00      0.00        27
      netral       0.95      1.00      0.97      1215
     positif       0.00      0.00      0.00        42

    accuracy                           0.95      1284
   macro avg       0.32      0.33      0.32      1284
weighted avg       0.90      0.95      0.92      1284


=== SVM ===
              precision    recall  f1-score   support

     negatif       1.00      0.33      0.50        27
      netral       0.98      1.00      0.99      1215
     positif       1.00      0.71      0.83        42

    accuracy                           0.98      1284
   macro avg       0.99      0.68      0.77      1284
weighted avg       0.98      0.98      0.97      1284


=== LOGISTIC ===
              precision    recall  f1-score   support

     negatif       1.00      0.11      0.20        27
      netral       0.96      1.00      0.98      1215
     positif       1.00

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
import pandas as pd

results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM', 'Logistic Regression'],
    'Accuracy': [acc_nb, acc_svm, acc_lr]
})

print(results.sort_values(by='Accuracy', ascending=False))

                 Model  Accuracy
1                  SVM  0.976636
2  Logistic Regression  0.958723
0          Naive Bayes  0.946262


In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'].fillna(''))

X_seq = tokenizer.texts_to_sequences(df['clean_text'].fillna(''))
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(df['label'])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y_enc, test_size=0.2, random_state=42
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=128),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

import numpy as np
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

acc_lstm = accuracy_score(y_test, y_pred)

print("LSTM Accuracy:", acc_lstm)
print(classification_report(y_test, y_pred))

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 86ms/step - accuracy: 0.9540 - loss: 0.2551 - val_accuracy: 0.9463 - val_loss: 0.2467
Epoch 2/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 20s 83ms/step - accuracy: 0.9575 - loss: 0.2064 - val_accuracy: 0.9455 - val_loss: 0.2501
Epoch 3/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 21s 86ms/step - accuracy: 0.9585 - loss: 0.2035 - val_accuracy: 0.9439 - val_loss: 0.2464
Epoch 4/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 14s 89ms/step - accuracy: 0.9587 - loss: 0.2029 - val_accuracy: 0.9463 - val_loss: 0.2484
Epoch 5/5
161/161 ━━━━━━━━━━━━━━━━━━━━ 20s 89ms/step - accuracy: 0.9589 - loss: 0.2010 - val_accuracy: 0.9463 - val_loss: 0.2530
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step
LSTM Accuracy: 0.9462616822429907
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        27
           1       0.95      1.00      0.97      1215
           2       0.00      0.00      0.00        42

    accuracy                           0.95      1284
   macro avg       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM', 'Logistic Regression', 'LSTM'],
    'Accuracy': [acc_nb, acc_svm, acc_lr, acc_lstm]
})

print(results.sort_values(by='Accuracy', ascending=False))

                 Model  Accuracy
1                  SVM  0.976636
2  Logistic Regression  0.958723
0          Naive Bayes  0.946262
3                 LSTM  0.946262
